# Retail KPI & Data Quality Profile

**Source files**
- `retail-orders-raw.csv`
- `retail-data-dictionary.csv`

**Decision owner:** Sales / Operations Manager  
**Refresh cadence:** Daily  
**Assessment date:** 2026-10-05

This notebook applies the business rules from the supplied data dictionary and creates an auditable KPI snapshot.


In [4]:
import pandas as pd
import numpy as np

# Load main retail orders data
df = pd.read_csv("retail-orders-raw.csv")

# Load data dictionary
data_dictionary = pd.read_csv("retail-data-dictionary.csv")

print("Data loaded successfully!")
print("Number of rows:", len(df))
print("Number of columns:", len(df.columns))

print("\nFirst 5 rows of retail data:")
display(df.head())

print("\nData Dictionary:")
display(data_dictionary)


Data loaded successfully!
Number of rows: 12
Number of columns: 9

First 5 rows of retail data:


,order_id,order_date,customer_segment,city,category,quantity,unit_price,discount_pct,payment_status
0,RT-1001,2026-01-03,Student,Chennai,Learning Kit,2,799,10.0,Paid
1,RT-1002,03/01/2026,Fresher,Bengaluru,Course Access,1,1499,0.0,paid
2,RT-1003,2026-01-05,student,Chennai,Course Access,1,1499,NaN,Pending
3,RT-1004,2026-01-07,Professional,Hyderabad,Learning Kit,3,799,5.0,Paid
4,RT-1004,2026-01-07,Professional,Hyderabad,Learning Kit,3,799,5.0,Paid



Data Dictionary:


,column,data_type,business_definition,quality_rule
0,order_id,string,Unique order identifier,Required and unique
1,order_date,date,Date the customer placed the order,Required ISO date between 2025-01-01 and today
2,customer_segment,category,Commercial customer segment,Student Fresher or Professional after normaliz...
3,city,string,Customer billing city,Required non-empty text
4,category,category,Product family,Learning Kit Course Access or Mentor Session
5,quantity,integer,Units purchased,Whole number greater than zero
6,unit_price,decimal,Price per unit before discount,Non-negative INR amount
7,discount_pct,decimal,Percentage discount applied,Between 0 and 100; missing means zero only aft...
8,payment_status,category,Latest order settlement state,Paid Pending Failed or Refunded after normaliz...


## 1. Profile Data

In [5]:
print("========== DATA PROFILE ==========")

print("Rows:", len(df))
print("Columns:", len(df.columns))

print("\nColumn Names:")
print(df.columns.tolist())

print("\nData Types:")
display(df.dtypes.to_frame("Data Type"))

print("\nStatistical Summary:")
display(df.describe(include="all"))

print("\nUnique Values in Each Column:")

for col in df.columns:
    print(f"{col}: {df[col].nunique(dropna=True)} unique values")

========== DATA PROFILE ==========
Rows: 12
Columns: 9

Column Names:
['order_id', 'order_date', 'customer_segment', 'city', 'category', 'quantity', 'unit_price', 'discount_pct', 'payment_status']

Data Types:


,Data Type
order_id,object
order_date,object
customer_segment,object
city,object
category,object
quantity,object
unit_price,int64
discount_pct,float64
payment_status,object



Statistical Summary:


,order_id,order_date,customer_segment,city,category,quantity,unit_price,discount_pct,payment_status
count,12,11,12,11,12,12,12.000000,11.000000,12
unique,11,10,4,7,3,5,NaN,NaN,5
top,RT-1004,2026-01-07,Student,Chennai,Learning Kit,1,NaN,NaN,Paid
freq,2,2,4,3,5,5,NaN,NaN,7
mean,NaN,NaN,NaN,NaN,NaN,NaN,1082.333333,13.636364,NaN
std,NaN,NaN,NaN,NaN,NaN,NaN,318.614425,30.748245,NaN
min,NaN,NaN,NaN,NaN,NaN,NaN,799.000000,0.000000,NaN
25%,NaN,NaN,NaN,NaN,NaN,NaN,799.000000,0.000000,NaN
50%,NaN,NaN,NaN,NaN,NaN,NaN,999.000000,5.000000,NaN
75%,NaN,NaN,NaN,NaN,NaN,NaN,1499.000000,10.000000,NaN



Unique Values in Each Column:
order_id: 11 unique values
order_date: 10 unique values
customer_segment: 4 unique values
city: 7 unique values
category: 3 unique values
quantity: 5 unique values
unit_price: 3 unique values
discount_pct: 5 unique values
payment_status: 5 unique values


## 2. Missing values Checks

In [6]:


print("========== MISSING VALUE CHECK ==========")

missing_count = df.isnull().sum()

missing_percentage = (
    df.isnull().sum() / len(df) * 100
).round(2)

missing_report = pd.DataFrame({
    "Missing Count": missing_count,
    "Missing Percentage": missing_percentage
})

display(missing_report)

print("\nColumns having missing values:")

display(
    missing_report[
        missing_report["Missing Count"] > 0
    ]
)

========== MISSING VALUE CHECK ==========


,Missing Count,Missing Percentage
order_id,0,0.00
order_date,1,8.33
customer_segment,0,0.00
city,1,8.33
category,0,0.00
quantity,0,0.00
unit_price,0,0.00
discount_pct,1,8.33
payment_status,0,0.00



Columns having missing values:


,Missing Count,Missing Percentage
order_date,1,8.33
city,1,8.33
discount_pct,1,8.33


## 3. Duplicates Checks

In [7]:
print("========== DUPLICATE CHECK ==========")

# Complete duplicate rows
duplicate_rows = df.duplicated().sum()

print("Complete duplicate rows:", duplicate_rows)

if duplicate_rows > 0:
    print("\nDuplicate rows:")
    display(df[df.duplicated(keep=False)])

# Duplicate Order IDs
print("\n========== ORDER ID UNIQUENESS ==========")

duplicate_order_ids = df[
    df["order_id"].duplicated(keep=False)
]

print(
    "Records with duplicate Order IDs:",
    len(duplicate_order_ids)
)

if len(duplicate_order_ids) > 0:
    display(
        duplicate_order_ids.sort_values("order_id")
    )


========== DUPLICATE CHECK ==========
Complete duplicate rows: 1

Duplicate rows:


,order_id,order_date,customer_segment,city,category,quantity,unit_price,discount_pct,payment_status
3,RT-1004,2026-01-07,Professional,Hyderabad,Learning Kit,3,799,5.0,Paid
4,RT-1004,2026-01-07,Professional,Hyderabad,Learning Kit,3,799,5.0,Paid



========== ORDER ID UNIQUENESS ==========
Records with duplicate Order IDs: 2


,order_id,order_date,customer_segment,city,category,quantity,unit_price,discount_pct,payment_status
3,RT-1004,2026-01-07,Professional,Hyderabad,Learning Kit,3,799,5.0,Paid
4,RT-1004,2026-01-07,Professional,Hyderabad,Learning Kit,3,799,5.0,Paid


## 4. Validity Checks

In [8]:
work = df.copy()

# Normalize text values
work["customer_segment_clean"] = (
    work["customer_segment"]
    .astype("string")
    .str.strip()
    .str.title()
)

work["payment_status_clean"] = (
    work["payment_status"]
    .astype("string")
    .str.strip()
    .str.title()
)

work["category_clean"] = (
    work["category"]
    .astype("string")
    .str.strip()
)

# Convert quantity to numeric
work["quantity_numeric"] = pd.to_numeric(
    work["quantity"],
    errors="coerce"
)

# Convert price to numeric
work["unit_price_numeric"] = pd.to_numeric(
    work["unit_price"],
    errors="coerce"
)

# Convert discount to numeric
work["discount_numeric"] = pd.to_numeric(
    work["discount_pct"],
    errors="coerce"
)

# STRICT ISO DATE CHECK
# Data dictionary requires YYYY-MM-DD
work["order_date_parsed"] = pd.to_datetime(
    work["order_date"],
    format="%Y-%m-%d",
    errors="coerce"
)

print("Prepared fields successfully.")

display(
    work[
        [
            "order_id",
            "order_date",
            "order_date_parsed",
            "quantity",
            "quantity_numeric",
            "unit_price",
            "unit_price_numeric",
            "discount_pct",
            "discount_numeric"
        ]
    ]
)


Prepared fields successfully.


,order_id,order_date,order_date_parsed,quantity,quantity_numeric,unit_price,unit_price_numeric,discount_pct,discount_numeric
0,RT-1001,2026-01-03,2026-01-03,2,2.0,799,799,10.0,10.0
1,RT-1002,03/01/2026,NaT,1,1.0,1499,1499,0.0,0.0
2,RT-1003,2026-01-05,2026-01-05,1,1.0,1499,1499,NaN,NaN
3,RT-1004,2026-01-07,2026-01-07,3,3.0,799,799,5.0,5.0
4,RT-1004,2026-01-07,2026-01-07,3,3.0,799,799,5.0,5.0
5,RT-1005,2026-01-09,2026-01-09,1,1.0,999,999,0.0,0.0
6,RT-1006,2026-13-10,NaT,-1,-1.0,799,799,10.0,10.0
7,RT-1007,2026-01-12,2026-01-12,2,2.0,999,999,105.0,105.0
8,RT-1008,2026-01-14,2026-01-14,two,NaN,1499,1499,0.0,0.0
9,RT-1009,2026-01-16,2026-01-16,1,1.0,799,799,0.0,0.0


In [9]:
# Allowed values according to data dictionary
allowed_segments = [
    "Student",
    "Fresher",
    "Professional"
]

allowed_categories = [
    "Learning Kit",
    "Course Access",
    "Mentor Session"
]

allowed_payment_status = [
    "Paid",
    "Pending",
    "Failed",
    "Refunded"
]

valid_order_id = work["order_id"].notna()

print("Missing Order IDs:",
      (~valid_order_id).sum())

valid_date = (
    work["order_date"].notna()
    & work["order_date_parsed"].notna()
)

print("Invalid/Missing Dates:",
      (~valid_date).sum())

if (~valid_date).sum() > 0:
    display(
        work[~valid_date][
            ["order_id", "order_date"]
        ]
    )

valid_segment = work[
    "customer_segment_clean"
].isin(allowed_segments)

print(
    "Invalid Customer Segments:",
    (~valid_segment).sum()
)

valid_city = (
    work["city"].notna()
    & work["city"].astype("string").str.strip().ne("")
)

print(
    "Missing/Invalid Cities:",
    (~valid_city).sum()
)

valid_category = work[
    "category_clean"
].isin(allowed_categories)

print(
    "Invalid Categories:",
    (~valid_category).sum()
)

valid_quantity = (
    work["quantity_numeric"].notna()
    & (work["quantity_numeric"] > 0)
    & (
        work["quantity_numeric"]
        == work["quantity_numeric"].round()
    )
)

print(
    "Invalid Quantities:",
    (~valid_quantity).sum()
)

if (~valid_quantity).sum() > 0:
    display(
        work[~valid_quantity][
            ["order_id", "quantity"]
        ]
    )

valid_price = (
    work["unit_price_numeric"].notna()
    & (work["unit_price_numeric"] >= 0)
)

print(
    "Invalid Unit Prices:",
    (~valid_price).sum()
)

valid_discount = (
    work["discount_numeric"].notna()
    & work["discount_numeric"].between(0, 100)
)

print(
    "Invalid/Missing Discounts:",
    (~valid_discount).sum()
)

if (~valid_discount).sum() > 0:
    display(
        work[~valid_discount][
            ["order_id", "discount_pct"]
        ]
    )

valid_payment = work[
    "payment_status_clean"
].isin(allowed_payment_status)

print(
    "Invalid Payment Status:",
    (~valid_payment).sum()
)


Missing Order IDs: 0
Invalid/Missing Dates: 3


,order_id,order_date
1,RT-1002,03/01/2026
6,RT-1006,2026-13-10
11,RT-1011,NaN


Invalid Customer Segments: 0
Missing/Invalid Cities: 1
Invalid Categories: 0
Invalid Quantities: 2


,order_id,quantity
6,RT-1006,-1
8,RT-1008,two


Invalid Unit Prices: 0
Invalid/Missing Discounts: 2


,order_id,discount_pct
2,RT-1003,NaN
7,RT-1007,105.0


Invalid Payment Status: 0


## 5. Consistency Checks

In [10]:
print("========== CONSISTENCY CHECK ==========")

# Core transaction fields must all be valid
core_consistent = (
    valid_order_id
    & valid_date
    & valid_quantity
    & valid_price
    & valid_discount
    & valid_payment
)

print(
    "Consistent Records:",
    core_consistent.sum()
)

print(
    "Inconsistent Records:",
    (~core_consistent).sum()
)

print("\nInconsistent Records:")

display(
    work[~core_consistent]
)

========== CONSISTENCY CHECK ==========
Consistent Records: 6
Inconsistent Records: 6

Inconsistent Records:


,order_id,order_date,customer_segment,city,category,quantity,unit_price,discount_pct,payment_status,customer_segment_clean,payment_status_clean,category_clean,quantity_numeric,unit_price_numeric,discount_numeric,order_date_parsed
1,RT-1002,03/01/2026,Fresher,Bengaluru,Course Access,1,1499,0.0,paid,Fresher,Paid,Course Access,1.0,1499,0.0,NaT
2,RT-1003,2026-01-05,student,Chennai,Course Access,1,1499,NaN,Pending,Student,Pending,Course Access,1.0,1499,NaN,2026-01-05
6,RT-1006,2026-13-10,Student,Pune,Learning Kit,-1,799,10.0,Paid,Student,Paid,Learning Kit,-1.0,799,10.0,NaT
7,RT-1007,2026-01-12,Professional,Mumbai,Mentor Session,2,999,105.0,Paid,Professional,Paid,Mentor Session,2.0,999,105.0,2026-01-12
8,RT-1008,2026-01-14,Fresher,Bengaluru,Course Access,two,1499,0.0,Pending,Fresher,Pending,Course Access,NaN,1499,0.0,2026-01-14
11,RT-1011,NaN,Student,Kochi,Mentor Session,1,999,0.0,Paid,Student,Paid,Mentor Session,1.0,999,0.0,NaT


## 6. Freshness

In [11]:
print("========== FRESHNESS CHECK ==========")

# Assessment date for the project
assessment_date = pd.Timestamp("2026-10-05")

# Find latest valid ISO date
latest_valid_date = work[
    "order_date_parsed"
].max()

# Calculate how old the data is
data_age_days = (
    assessment_date - latest_valid_date
).days

print(
    "Latest Valid Order Date:",
    latest_valid_date.date()
)

print(
    "Assessment Date:",
    assessment_date.date()
)

print(
    "Data Age:",
    data_age_days,
    "days"
)

# Requirement: data should not be older than 1 day
freshness_threshold = 1

if data_age_days <= freshness_threshold:
    freshness_status = "PASS"
else:
    freshness_status = "FAIL"

print(
    "Freshness Status:",
    freshness_status
)


========== FRESHNESS CHECK ==========
Latest Valid Order Date: 2026-01-18
Assessment Date: 2026-10-05
Data Age: 260 days
Freshness Status: FAIL


## 7. Final Data Quality Contract Test

In [12]:
quality_report = pd.DataFrame({

    "Quality Check": [
        "Order ID Completeness",
        "Order ID Uniqueness",
        "Order Date Completeness",
        "Order Date Validity",
        "City Completeness",
        "Customer Segment Validity",
        "Category Validity",
        "Quantity Validity",
        "Unit Price Validity",
        "Discount Validity",
        "Payment Status Validity",
        "Core Consistency",
        "Freshness"
    ],

    "Pass Rate (%)": [

        valid_order_id.mean() * 100,

        (
            work["order_id"].nunique()
            / work["order_id"].notna().sum()
        ) * 100,

        work["order_date"].notna().mean() * 100,

        valid_date.mean() * 100,

        valid_city.mean() * 100,

        valid_segment.mean() * 100,

        valid_category.mean() * 100,

        valid_quantity.mean() * 100,

        valid_price.mean() * 100,

        valid_discount.mean() * 100,

        valid_payment.mean() * 100,

        core_consistent.mean() * 100,

        100 if freshness_status == "PASS" else 0
    ],

    "Threshold (%)": [
        100,
        100,
        98,
        98,
        99,
        100,
        100,
        98,
        100,
        98,
        100,
        98,
        100
    ]
})

# Determine PASS / FAIL
quality_report["Status"] = np.where(
    quality_report["Pass Rate (%)"]
    >= quality_report["Threshold (%)"],
    "PASS",
    "FAIL"
)

quality_report["Pass Rate (%)"] = (
    quality_report["Pass Rate (%)"]
    .round(2)
)

display(quality_report)


,Quality Check,Pass Rate (%),Threshold (%),Status
0,Order ID Completeness,100.00,100,PASS
1,Order ID Uniqueness,91.67,100,FAIL
2,Order Date Completeness,91.67,98,FAIL
3,Order Date Validity,75.00,98,FAIL
4,City Completeness,91.67,99,FAIL
5,Customer Segment Validity,100.00,100,PASS
6,Category Validity,100.00,100,PASS
7,Quantity Validity,83.33,98,FAIL
8,Unit Price Validity,100.00,100,PASS
9,Discount Validity,83.33,98,FAIL


In [13]:
print("----OVERALL DATA QUALITY STATUS----")
failed_checks = quality_report[
    quality_report["Status"] == "FAIL"
]

print("========== OVERALL RESULT ==========")

print(
    "Total Quality Checks:",
    len(quality_report)
)

print(
    "Passed Checks:",
    (quality_report["Status"] == "PASS").sum()
)

print(
    "Failed Checks:",
    (quality_report["Status"] == "FAIL").sum()
)

if len(failed_checks) == 0:
    overall_status = "PASS"
else:
    overall_status = "FAIL"

print("\nOVERALL DATA QUALITY STATUS:",
      overall_status)

print("\nFailed Checks:")

display(failed_checks)

----OVERALL DATA QUALITY STATUS----
========== OVERALL RESULT ==========
Total Quality Checks: 13
Passed Checks: 5
Failed Checks: 8

OVERALL DATA QUALITY STATUS: FAIL

Failed Checks:


,Quality Check,Pass Rate (%),Threshold (%),Status
1,Order ID Uniqueness,91.67,100,FAIL
2,Order Date Completeness,91.67,98,FAIL
3,Order Date Validity,75.00,98,FAIL
4,City Completeness,91.67,99,FAIL
7,Quantity Validity,83.33,98,FAIL
9,Discount Validity,83.33,98,FAIL
11,Core Consistency,50.00,98,FAIL
12,Freshness,0.00,100,FAIL


## 8. KPI Calculation

In [14]:
# Remove exact duplicate rows ONLY for KPI calculation.
# The original df is NOT modified.

dedup = work.drop_duplicates().copy()

# KPI eligibility rules
kpi_eligible = dedup[
    dedup["order_id"].notna()
    & dedup["order_date_parsed"].notna()
    & dedup["quantity_numeric"].notna()
    & (dedup["quantity_numeric"] > 0)
    & (
        dedup["quantity_numeric"]
        == dedup["quantity_numeric"].round()
    )
    & dedup["unit_price_numeric"].notna()
    & (dedup["unit_price_numeric"] >= 0)
    & dedup["discount_numeric"].notna()
    & dedup["discount_numeric"].between(0, 100)
    & dedup["payment_status_clean"].isin(
        allowed_payment_status
    )
].copy()

print("Raw Records:",
      len(work))

print(
    "KPI-Eligible Records:",
    len(kpi_eligible)
)

print(
    "Excluded Records:",
    len(work) - len(kpi_eligible)
)

print("\nKPI-Eligible Order IDs:")

print(
    kpi_eligible["order_id"].tolist()
)

Raw Records: 12
KPI-Eligible Records: 5
Excluded Records: 7

KPI-Eligible Order IDs:
['RT-1001', 'RT-1004', 'RT-1005', 'RT-1009', 'RT-1010']


In [15]:
print("---- KPI SALES FIELDS ----")
# Gross sales
kpi_eligible["gross_sales"] = (
    kpi_eligible["quantity_numeric"]
    * kpi_eligible["unit_price_numeric"]
)

# Discount amount
kpi_eligible["discount_amount"] = (
    kpi_eligible["gross_sales"]
    * kpi_eligible["discount_numeric"]
    / 100
)

# Net sales
kpi_eligible["net_sales"] = (
    kpi_eligible["gross_sales"]
    - kpi_eligible["discount_amount"]
)

# Payment status flags
kpi_eligible["is_paid"] = (
    kpi_eligible["payment_status_clean"]
    == "Paid"
)

kpi_eligible["is_refunded"] = (
    kpi_eligible["payment_status_clean"]
    == "Refunded"
)

kpi_eligible["is_failed"] = (
    kpi_eligible["payment_status_clean"]
    == "Failed"
)

kpi_eligible["is_pending"] = (
    kpi_eligible["payment_status_clean"]
    == "Pending"
)

display(
    kpi_eligible[
        [
            "order_id",
            "quantity_numeric",
            "unit_price_numeric",
            "discount_numeric",
            "gross_sales",
            "discount_amount",
            "net_sales",
            "payment_status_clean"
        ]
    ]
)

---- KPI SALES FIELDS ----


,order_id,quantity_numeric,unit_price_numeric,discount_numeric,gross_sales,discount_amount,net_sales,payment_status_clean
0,RT-1001,2.0,799,10.0,1598.0,159.80,1438.20,Paid
3,RT-1004,3.0,799,5.0,2397.0,119.85,2277.15,Paid
5,RT-1005,1.0,999,0.0,999.0,0.00,999.00,Failed
9,RT-1009,1.0,799,0.0,799.0,0.00,799.00,Paid
10,RT-1010,2.0,1499,15.0,2998.0,449.70,2548.30,Refunded


In [16]:
print("----- CALCULATE 10 KPIs -----")

# KPI 1: Total Orders
total_orders = (
    kpi_eligible["order_id"].nunique()
)

# KPI 2: Gross Sales
gross_sales = (
    kpi_eligible["gross_sales"].sum()
)

# KPI 3: Discount Amount
discount_amount = (
    kpi_eligible["discount_amount"].sum()
)

# Paid orders
paid_orders = (
    kpi_eligible.loc[
        kpi_eligible["is_paid"],
        "order_id"
    ].nunique()
)

# KPI 4: Recognized Net Sales
recognized_net_sales = (
    kpi_eligible.loc[
        kpi_eligible["is_paid"],
        "net_sales"
    ].sum()
)

# KPI 5: Units Sold
units_sold = (
    kpi_eligible["quantity_numeric"].sum()
)

# KPI 6: Average Order Value
aov = (
    recognized_net_sales / paid_orders
    if paid_orders > 0
    else 0
)

# KPI 7: Paid Order Rate
paid_order_rate = (
    paid_orders / total_orders * 100
    if total_orders > 0
    else 0
)

# KPI 8: Refund Rate
refund_rate = (
    kpi_eligible["is_refunded"].sum()
    / total_orders * 100
    if total_orders > 0
    else 0
)

# KPI 9: Failed Payment Rate
failed_payment_rate = (
    kpi_eligible["is_failed"].sum()
    / total_orders * 100
    if total_orders > 0
    else 0
)

# KPI 10: Pending Payment Rate
pending_payment_rate = (
    kpi_eligible["is_pending"].sum()
    / total_orders * 100
    if total_orders > 0
    else 0
)

# Create KPI report
kpi_report = pd.DataFrame({

    "KPI": [
        "Total Orders",
        "Gross Sales",
        "Discount Amount",
        "Recognized Net Sales",
        "Units Sold",
        "Average Order Value",
        "Paid Order Rate",
        "Refund Rate",
        "Failed Payment Rate",
        "Pending Payment Rate"
    ],

    "Formula": [
        "COUNT(DISTINCT order_id)",
        "SUM(quantity × unit_price)",
        "SUM(gross_sales × discount_pct / 100)",
        "SUM(net_sales) for Paid orders",
        "SUM(quantity)",
        "Recognized Net Sales ÷ Paid Orders",
        "Paid Orders ÷ Total Orders × 100",
        "Refunded Orders ÷ Total Orders × 100",
        "Failed Orders ÷ Total Orders × 100",
        "Pending Orders ÷ Total Orders × 100"
    ],

    "Value": [
        total_orders,
        gross_sales,
        discount_amount,
        recognized_net_sales,
        units_sold,
        aov,
        paid_order_rate,
        refund_rate,
        failed_payment_rate,
        pending_payment_rate
    ]
})

display(kpi_report)

----- CALCULATE 10 KPIs -----


,KPI,Formula,Value
0,Total Orders,COUNT(DISTINCT order_id),5.000000
1,Gross Sales,SUM(quantity × unit_price),8791.000000
2,Discount Amount,SUM(gross_sales × discount_pct / 100),729.350000
3,Recognized Net Sales,SUM(net_sales) for Paid orders,4514.350000
4,Units Sold,SUM(quantity),9.000000
5,Average Order Value,Recognized Net Sales ÷ Paid Orders,1504.783333
6,Paid Order Rate,Paid Orders ÷ Total Orders × 100,60.000000
7,Refund Rate,Refunded Orders ÷ Total Orders × 100,20.000000
8,Failed Payment Rate,Failed Orders ÷ Total Orders × 100,20.000000
9,Pending Payment Rate,Pending Orders ÷ Total Orders × 100,0.000000


## 9. Conclusion

In [18]:
# ==========================================
print("--- FINAL PROJECT SUMMARY ---")
# ==========================================

print("=" * 60)
print("       RETAIL KPI & DATA QUALITY PROJECT")
print("=" * 60)

print("\nDATASET")
print("Total Raw Records:", len(df))
print("Total Columns:", len(df.columns))

print("\nDATA QUALITY")
print(
    "Passed Checks:",
    (quality_report["Status"] == "PASS").sum()
)

print(
    "Failed Checks:",
    (quality_report["Status"] == "FAIL").sum()
)

print(
    "Overall Data Quality:",
    overall_status
)

print("\nKPI DATA")
print(
    "KPI-Eligible Records:",
    len(kpi_eligible)
)

print(
    "Excluded Records:",
    len(df) - len(kpi_eligible)
)

print("\nIMPORTANT DATA ISSUES")

if len(failed_checks) > 0:
    for check in failed_checks["Quality Check"]:
        print("❌", check)
else:
    print("No quality issues found.")

print("\nKPI RESULTS")
display(kpi_report)

print("\nFINAL CONCLUSION")

if overall_status == "FAIL":

    print(
        "The dataset does NOT currently meet "
        "the defined data-quality contract."
    )

    print(
        "The identified issues must be investigated "
        "before production KPI reporting."
    )

else:

    print(
        "The dataset meets all defined "
        "data-quality requirements."
    )

--- FINAL PROJECT SUMMARY ---
       RETAIL KPI & DATA QUALITY PROJECT

DATASET
Total Raw Records: 12
Total Columns: 9

DATA QUALITY
Passed Checks: 5
Failed Checks: 8
Overall Data Quality: FAIL

KPI DATA
KPI-Eligible Records: 5
Excluded Records: 7

IMPORTANT DATA ISSUES
❌ Order ID Uniqueness
❌ Order Date Completeness
❌ Order Date Validity
❌ City Completeness
❌ Quantity Validity
❌ Discount Validity
❌ Core Consistency
❌ Freshness

KPI RESULTS


,KPI,Formula,Value
0,Total Orders,COUNT(DISTINCT order_id),5.000000
1,Gross Sales,SUM(quantity × unit_price),8791.000000
2,Discount Amount,SUM(gross_sales × discount_pct / 100),729.350000
3,Recognized Net Sales,SUM(net_sales) for Paid orders,4514.350000
4,Units Sold,SUM(quantity),9.000000
5,Average Order Value,Recognized Net Sales ÷ Paid Orders,1504.783333
6,Paid Order Rate,Paid Orders ÷ Total Orders × 100,60.000000
7,Refund Rate,Refunded Orders ÷ Total Orders × 100,20.000000
8,Failed Payment Rate,Failed Orders ÷ Total Orders × 100,20.000000
9,Pending Payment Rate,Pending Orders ÷ Total Orders × 100,0.000000



FINAL CONCLUSION
The dataset does NOT currently meet the defined data-quality contract.
The identified issues must be investigated before production KPI reporting.
